# Hệ gợi ý việc làm trực tuyến
## Giai đoạn 1 và 2 — thực hành bằng Jupyter Notebook

**Mục tiêu:** gợi ý việc làm tương tự tin đang xem và tìm việc phù hợp với văn bản CV.
Notebook sử dụng Python, pandas và thiết kế hướng đối tượng từ các module của dự án.

**Cách chạy:** tải toàn bộ repository, mở notebook tại thư mục gốc, chọn kernel Python
của môi trường đã cài thư viện, rồi chọn **Restart Kernel and Run All Cells** hoặc chạy
từng ô bằng **Shift + Enter**. Các ô Markdown giải thích phương pháp; các ô Code thực thi.
Không chỉ tải riêng file `.ipynb`: notebook cần `generate_mock_data.py` và `job_recommender.py`.

Mặc định dùng **TF-IDF**, không tải mô hình từ mạng. Đây là demo Giai đoạn 1–2;
chưa triển khai các metric đánh giá của Giai đoạn 3 hay Streamlit của Giai đoạn 4.


### 1. Nạp thư viện và chọn cấu hình

Cài môi trường theo README trước khi chạy. Nếu báo thiếu module, kiểm tra kernel đã chọn.
`LANGUAGE = "vi"` sử dụng tin và CV tiếng Việt; đổi thành `"en"` để dùng dữ liệu tiếng Anh.
Mỗi lần đổi cấu hình, chạy lại các ô bên dưới theo thứ tự.


In [1]:
from pathlib import Path
import json
import inspect

import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Code
from sklearn.metrics.pairwise import cosine_similarity

# Notebook đặt cùng thư mục với hai module này để tái sử dụng mã OOP đã viết.
from generate_mock_data import MockDataGenerator, JOB_COLUMNS
from job_recommender import JobRecommenderSystem, TextPreprocessor, TfidfTextVectorizer

LANGUAGE = "vi"                  # "vi" hoặc "en"
BACKEND = "tfidf"                # Hoặc "sentence-transformers" sau khi cài thêm thư viện
JOB_ID = "J001"                  # Tin đang xem: chuyên viên phân tích dữ liệu
TOP_N = 5

if LANGUAGE not in {"vi", "en"}:
    raise ValueError("LANGUAGE phải là 'vi' hoặc 'en'.")

DATA_DIR = Path("data")
pd.set_option("display.max_colwidth", 100)
print(f"Ngôn ngữ: {LANGUAGE}; backend: {BACKEND}; top_n: {TOP_N}")


Ngôn ngữ: vi; backend: tfidf; top_n: 5


### 2. Giai đoạn 1 — tạo và đọc dữ liệu giả lập

`MockDataGenerator` tạo 20 việc làm bằng pandas. Bộ dữ liệu có 6 cột:
`Job_ID`, `Title`, `Skills`, `Experience_Years`, `Domain`, `Job_Description`.
`Experience_Years` là số năm kinh nghiệm tối thiểu trong tin; `Domain` là nhóm chuyên môn IT.

File tiếng Việt giữ tên **jobs_vie.cvs** theo quy ước của dự án; nội dung vẫn là CSV,
vì vậy đọc bằng `pd.read_csv`. Hai bản ngôn ngữ có cùng Job_ID, cần sử dụng riêng.
Ô dưới chỉ tạo file chưa tồn tại, bảo toàn những file dữ liệu bạn đã sửa.


In [2]:
generator = MockDataGenerator()
DATA_DIR.mkdir(parents=True, exist_ok=True)
jobs_path = DATA_DIR / ("jobs_vie.cvs" if LANGUAGE == "vi" else "jobs.csv")

if not jobs_path.exists():
    generated_jobs = generator.create_vietnamese_jobs() if LANGUAGE == "vi" else generator.create_jobs()
    generated_jobs.to_csv(jobs_path, index=False, encoding="utf-8-sig")

jobs = pd.read_csv(jobs_path, encoding="utf-8-sig", dtype={"Job_ID": str})
assert list(jobs.columns) == JOB_COLUMNS
assert len(jobs) >= 15 and jobs["Job_ID"].is_unique
assert jobs.notna().all().all()

print(f"Đã đọc {len(jobs)} việc làm từ {jobs_path.as_posix()}")
display(jobs)


Đã đọc 20 việc làm từ data/jobs_vie.cvs


,Job_ID,Title,Skills,Experience_Years,Domain,Job_Description
0,J001,Chuyên viên phân tích dữ liệu,SQL; Python; Pandas; Excel; Power BI,1,Phân tích dữ liệu,"Phân tích dữ liệu bán hàng và khách hàng bằng SQL, Python và Pandas. Làm sạch dữ liệu, xây dựng ..."
1,J002,Chuyên viên phân tích dữ liệu kinh doanh,SQL; Power BI; Tableau; Excel; DAX,2,Phân tích dữ liệu,"Xây dựng bảng điều khiển phân tích kinh doanh và báo cáo bán hàng bằng Power BI, DAX và SQL. Xác..."
2,J003,Nhà khoa học dữ liệu,Python; Pandas; SQL; Thống kê; Scikit-learn,2,Trí tuệ nhân tạo,"Khám phá dữ liệu khách hàng bằng Python và Pandas. Áp dụng phương pháp thống kê, huấn luyện mô h..."
3,J004,Kỹ sư dữ liệu,Python; SQL; Spark; Airflow; ETL,2,Kỹ thuật dữ liệu,"Phát triển quy trình trích xuất, biến đổi và nạp dữ liệu ETL bằng Python, SQL, Spark và Airflow...."
4,J005,Kỹ sư trí tuệ nhân tạo,Python; PyTorch; Transformers; Xử lý ngôn ngữ tự nhiên (NLP); FastAPI,2,Trí tuệ nhân tạo,Xây dựng ứng dụng xử lý ngôn ngữ tự nhiên bằng Transformers và PyTorch. Phát triển mô hình tìm k...
5,J006,Kỹ sư học máy,Python; Scikit-learn; PyTorch; Docker; MLflow,3,Trí tuệ nhân tạo,"Huấn luyện và triển khai mô hình học máy bằng Python, Scikit-learn và PyTorch. Theo dõi thử nghi..."
6,J007,Kỹ sư xử lý ngôn ngữ tự nhiên,Python; Xử lý ngôn ngữ tự nhiên (NLP); Transformers; PyTorch; Phân loại văn bản,2,Trí tuệ nhân tạo,"Xử lý văn bản và xây dựng mô hình xử lý ngôn ngữ tự nhiên để phân loại văn bản, tìm kiếm ngữ ngh..."
7,J008,Kỹ sư thị giác máy tính,Python; PyTorch; OpenCV; Học sâu; CNN,2,Trí tuệ nhân tạo,Xây dựng hệ thống phân loại ảnh và phát hiện đối tượng. Chuẩn bị dữ liệu hình ảnh bằng OpenCV và...
8,J009,Lập trình viên Backend Python,Python; FastAPI; Django; PostgreSQL; REST API,2,Phát triển phần mềm,"Phát triển dịch vụ REST API bằng Python, FastAPI và Django. Thiết kế cơ sở dữ liệu PostgreSQL, t..."
9,J010,Lập trình viên Backend Java,Java; Spring Boot; PostgreSQL; REST API; Docker,2,Phát triển phần mềm,Phát triển các vi dịch vụ phía máy chủ và REST API bằng Java và Spring Boot. Tối ưu truy vấn Pos...


### 3. Chuẩn bị ba CV để chạy thử

Ba hồ sơ giả lập lần lượt thuộc nhóm phân tích dữ liệu, backend Python và AI/NLP.
CV được viết cùng ngôn ngữ với tin vì TF-IDF không tự dịch từ ngữ.
Đây là dữ liệu minh họa, không phải hồ sơ ứng viên thật.


In [3]:
resumes_vi = [
    {
        "Resume_ID": "CV001", "Profile": "Ứng viên phân tích dữ liệu",
        "CV_Text": "Tôi có 1 năm kinh nghiệm phân tích dữ liệu. Tôi sử dụng SQL, Python và Pandas "
                   "để làm sạch và phân tích dữ liệu bán hàng, khách hàng. Tôi xây dựng bảng điều "
                   "khiển Power BI, báo cáo Excel và giải thích xu hướng kinh doanh, chỉ số KPI. "
                   "Tôi mong muốn làm chuyên viên phân tích dữ liệu hoặc phân tích dữ liệu kinh doanh."
    },
    {
        "Resume_ID": "CV002", "Profile": "Ứng viên backend Python",
        "CV_Text": "Tôi là lập trình viên Backend Python có 2 năm kinh nghiệm phát triển phần mềm. "
                   "Tôi xây dựng dịch vụ REST API bằng FastAPI và Django, thiết kế cơ sở dữ liệu "
                   "PostgreSQL và triển khai xác thực người dùng. Tôi viết kiểm thử đơn vị, "
                   "đóng gói dịch vụ bằng Docker và phối hợp với lập trình viên giao diện."
    },
    {
        "Resume_ID": "CV003", "Profile": "Ứng viên AI và xử lý ngôn ngữ",
        "CV_Text": "Tôi là kỹ sư trí tuệ nhân tạo có 2 năm kinh nghiệm. Tôi sử dụng Python, "
                   "PyTorch và Transformers cho xử lý ngôn ngữ tự nhiên, phân loại văn bản. "
                   "Tôi đã xây dựng tìm kiếm ngữ nghĩa, véc-tơ biểu diễn tài liệu và hệ gợi ý. "
                   "Tôi triển khai dịch vụ dự đoán của mô hình học máy bằng FastAPI và Docker."
    },
]

if LANGUAGE == "vi":
    resumes = resumes_vi
else:
    resumes_path = DATA_DIR / "resumes.json"
    resumes = (
        json.loads(resumes_path.read_text(encoding="utf-8"))
        if resumes_path.exists() else generator.create_resumes()
    )

display(pd.DataFrame(resumes))


,Resume_ID,Profile,CV_Text
0,CV001,Ứng viên phân tích dữ liệu,"Tôi có 1 năm kinh nghiệm phân tích dữ liệu. Tôi sử dụng SQL, Python và Pandas để làm sạch và phâ..."
1,CV002,Ứng viên backend Python,Tôi là lập trình viên Backend Python có 2 năm kinh nghiệm phát triển phần mềm. Tôi xây dựng dịch...
2,CV003,Ứng viên AI và xử lý ngôn ngữ,"Tôi là kỹ sư trí tuệ nhân tạo có 2 năm kinh nghiệm. Tôi sử dụng Python, PyTorch và Transformers ..."


### 4. Giai đoạn 2 — từ văn bản đến điểm tương đồng

Quy trình: **ghép nội dung tin → chuẩn hóa → vector hóa → cosine similarity → xếp hạng**.

- Ghép chức danh, kỹ năng, kinh nghiệm, lĩnh vực và mô tả; bỏ Job_ID khỏi đặc trưng.
- TF-IDF dùng unigram/bigram, học từ vựng từ tin tuyển dụng. CV chỉ được `transform`,
  không `fit` lại, để giữ cùng không gian vector.
- Sentence Transformers dùng mô hình pretrained để biểu diễn ngữ nghĩa.

$$\mathrm{cosine}(q,j) = \frac{q \cdot j}{\|q\|_2\,\|j\|_2}$$

Điểm lớn hơn biểu thị nội dung gần nhau hơn. Điểm không phải xác suất trúng tuyển.
Kinh nghiệm hiện là tín hiệu văn bản, chưa được kiểm tra như điều kiện tuyển dụng bắt buộc.
Ô tiếp theo minh họa công thức với hai văn bản ngắn, không phải evaluation mô hình.


In [4]:
example_texts = [
    "SQL Python Pandas phân tích dữ liệu",
    "SQL Power BI phân tích dữ liệu kinh doanh",
]
example_encoder = TfidfTextVectorizer()
example_vectors = example_encoder.fit_transform(example_texts)
example_score = cosine_similarity(example_vectors[0:1], example_vectors[1:2])[0, 0]

print("Văn bản 1:", TextPreprocessor.normalize(example_texts[0]))
print("Văn bản 2:", TextPreprocessor.normalize(example_texts[1]))
print("Kích thước ma trận TF-IDF:", example_vectors.shape)
print(f"Cosine similarity: {example_score:.4f}")


Văn bản 1: SQL Python Pandas phân tích dữ liệu
Văn bản 2: SQL Power BI phân tích dữ liệu kinh doanh
Kích thước ma trận TF-IDF: (2, 22)
Cosine similarity: 0.3727


### 5. Xem mã nguồn OOP và khởi tạo AI Engine

Notebook tái sử dụng `JobRecommenderSystem` trong `job_recommender.py` để kết quả
nhất quán với chương trình Python. Các phương thức bên dưới là mã nguồn thật đang được chạy.
Bạn có thể mở module để xem đầy đủ bước kiểm tra dữ liệu, tiền xử lý và vector hóa.
Khi sửa module trong lúc kernel đang chạy, hãy khởi động lại kernel và chạy lại toàn bộ notebook.


In [5]:
for method in (
    JobRecommenderSystem.recommend_similar_jobs,
    JobRecommenderSystem.match_cv_to_jobs,
):
    display(Code(inspect.getsource(method), language="python"))


def recommend_similar_jobs(self, job_id: str, top_n: int = 5) -> pd.DataFrame:
        """So một việc làm với tất cả việc làm khác, loại chính nó khỏi top-N.

        Raises:
            KeyError: Job_ID không tồn tại.
            ValueError: top_n không phải số nguyên dương.
        """
        self._validate_top_n(top_n)
        if not isinstance(job_id, str):
            raise TypeError("job_id phải là chuỗi, ví dụ 'J001'.")
        job_id = TextPreprocessor.normalize(job_id)
        if job_id not in self._id_to_position:
            raise KeyError(f"Không tìm thấy Job_ID: {job_id}")
        position = self._id_to_position[job_id]
        # Chỉ tính vector 1 x N, tránh lưu ma trận N x N không cần thiết.
        query_vector = self._job_vectors[position:position + 1]
        scores = cosine_similarity(query_vector, self._job_vectors).ravel()
        return self._rank(scores, top_n, exclude_position=position)

def match_cv_to_jobs(self, cv_text: str, top_n: int = 5) -> pd.DataFrame:
        """Vector hóa toàn văn CV rồi xếp hạng độ tương đồng với mỗi tin.

        Không dùng regex để suy đoán số năm kinh nghiệm/thuộc tính cá nhân.
        Đây là trích xuất đặc trưng văn bản, chưa phải trích xuất trường CV.
        CV rỗng bị từ chối. Với TF-IDF, CV không có từ nào trong vocabulary
        trả bảng rỗng kèm cảnh báo thay vì gợi ý tùy ý từ các điểm đều bằng 0.
        """
        self._validate_top_n(top_n)
        if not isinstance(cv_text, str) or not cv_text.strip():
            raise ValueError("cv_text phải là chuỗi văn bản không rỗng.")
        query_vector = self._encoder.transform([TextPreprocessor.normalize(cv_text)])
        if self.backend == "tfidf" and query_vector.nnz == 0:
            warnings.warn(
                "CV không có từ thuộc vocabulary TF-IDF; không có đủ tín hiệu để gợi ý.",
                UserWarning, stacklevel=2,
            )
            return self._rank(np.zeros(len(self._jobs)), top_n).iloc[:0].copy()
        scores = cosine_similarity(query_vector, self._job_vectors).ravel()
        return self._rank(scores, top_n)

In [6]:
# Tính vector việc làm một lần, sau đó dùng lại cho các truy vấn.
engine = JobRecommenderSystem(jobs, backend=BACKEND)
print(f"Engine đã sẵn sàng với {len(jobs)} việc làm; backend={engine.backend}.")


Engine đã sẵn sàng với 20 việc làm; backend=tfidf.


### 6. Gợi ý việc làm tương tự tin đang xem (Item-to-Item)

`recommend_similar_jobs(job_id, top_n=5)` so sánh vector tin đang xem với các tin
khác, loại chính tin đó rồi lấy top-N theo điểm giảm dần. Nếu bằng điểm, dùng Job_ID
để giữ thứ tự ổn định. Đây là gợi ý dựa trên nội dung, chưa dùng lịch sử tương tác.


In [7]:
print("Tin tuyển dụng đang xem:")
display(jobs.loc[jobs["Job_ID"] == JOB_ID])

similar_jobs = engine.recommend_similar_jobs(JOB_ID, top_n=TOP_N)
result_columns = ["Rank", "Job_ID", "Title", "Skills", "Experience_Years", "Domain", "Similarity_Score"]
print("Những việc làm tương tự:")
display(similar_jobs[result_columns].round({"Similarity_Score": 4}))


Tin tuyển dụng đang xem:


,Job_ID,Title,Skills,Experience_Years,Domain,Job_Description
0,J001,Chuyên viên phân tích dữ liệu,SQL; Python; Pandas; Excel; Power BI,1,Phân tích dữ liệu,"Phân tích dữ liệu bán hàng và khách hàng bằng SQL, Python và Pandas. Làm sạch dữ liệu, xây dựng ..."


Những việc làm tương tự:


,Rank,Job_ID,Title,Skills,Experience_Years,Domain,Similarity_Score
0,1,J002,Chuyên viên phân tích dữ liệu kinh doanh,SQL; Power BI; Tableau; Excel; DAX,2,Phân tích dữ liệu,0.4881
1,2,J003,Nhà khoa học dữ liệu,Python; Pandas; SQL; Thống kê; Scikit-learn,2,Trí tuệ nhân tạo,0.1880
2,3,J004,Kỹ sư dữ liệu,Python; SQL; Spark; Airflow; ETL,2,Kỹ thuật dữ liệu,0.1431
3,4,J018,Chuyên viên phân tích an ninh mạng,Python; Linux; SIEM; Mạng máy tính; Bảo mật,2,An ninh mạng,0.1109
4,5,J019,Quản trị viên cơ sở dữ liệu,SQL; PostgreSQL; MySQL; Linux; Sao lưu dữ liệu,3,Quản trị cơ sở dữ liệu,0.0938


### 7. Ghép CV với việc làm (User-to-Item)

`match_cv_to_jobs(cv_text, top_n=5)` biến toàn văn CV thành vector và tìm các tin
có cosine similarity cao nhất. Chức năng này trích xuất đặc trưng văn bản, chưa
tách số năm kinh nghiệm hay kỹ năng thành các trường CV có cấu trúc.


In [8]:
all_matches = []
for resume in resumes:
    display(Markdown(f"**{resume['Resume_ID']} — {resume['Profile']}**"))
    print(resume["CV_Text"])
    matches = engine.match_cv_to_jobs(resume["CV_Text"], top_n=TOP_N)
    display(matches[result_columns].round({"Similarity_Score": 4}))
    all_matches.append(matches.assign(Resume_ID=resume["Resume_ID"]))

cv_matches = pd.concat(all_matches, ignore_index=True)


**CV001 — Ứng viên phân tích dữ liệu**

Tôi có 1 năm kinh nghiệm phân tích dữ liệu. Tôi sử dụng SQL, Python và Pandas để làm sạch và phân tích dữ liệu bán hàng, khách hàng. Tôi xây dựng bảng điều khiển Power BI, báo cáo Excel và giải thích xu hướng kinh doanh, chỉ số KPI. Tôi mong muốn làm chuyên viên phân tích dữ liệu hoặc phân tích dữ liệu kinh doanh.


,Rank,Job_ID,Title,Skills,Experience_Years,Domain,Similarity_Score
0,1,J001,Chuyên viên phân tích dữ liệu,SQL; Python; Pandas; Excel; Power BI,1,Phân tích dữ liệu,0.7818
1,2,J002,Chuyên viên phân tích dữ liệu kinh doanh,SQL; Power BI; Tableau; Excel; DAX,2,Phân tích dữ liệu,0.5435
2,3,J003,Nhà khoa học dữ liệu,Python; Pandas; SQL; Thống kê; Scikit-learn,2,Trí tuệ nhân tạo,0.1591
3,4,J004,Kỹ sư dữ liệu,Python; SQL; Spark; Airflow; ETL,2,Kỹ thuật dữ liệu,0.1285
4,5,J018,Chuyên viên phân tích an ninh mạng,Python; Linux; SIEM; Mạng máy tính; Bảo mật,2,An ninh mạng,0.1118


**CV002 — Ứng viên backend Python**

Tôi là lập trình viên Backend Python có 2 năm kinh nghiệm phát triển phần mềm. Tôi xây dựng dịch vụ REST API bằng FastAPI và Django, thiết kế cơ sở dữ liệu PostgreSQL và triển khai xác thực người dùng. Tôi viết kiểm thử đơn vị, đóng gói dịch vụ bằng Docker và phối hợp với lập trình viên giao diện.


,Rank,Job_ID,Title,Skills,Experience_Years,Domain,Similarity_Score
0,1,J009,Lập trình viên Backend Python,Python; FastAPI; Django; PostgreSQL; REST API,2,Phát triển phần mềm,0.6416
1,2,J011,Lập trình viên Backend Node.js,JavaScript; TypeScript; Node.js; Express; PostgreSQL,1,Phát triển phần mềm,0.3664
2,3,J020,Lập trình viên Backend .NET,C#; .NET; ASP.NET; SQL Server; REST API,2,Phát triển phần mềm,0.3593
3,4,J013,Lập trình viên Full Stack,TypeScript; React; Node.js; PostgreSQL; REST API,3,Phát triển phần mềm,0.3090
4,5,J010,Lập trình viên Backend Java,Java; Spring Boot; PostgreSQL; REST API; Docker,2,Phát triển phần mềm,0.2973


**CV003 — Ứng viên AI và xử lý ngôn ngữ**

Tôi là kỹ sư trí tuệ nhân tạo có 2 năm kinh nghiệm. Tôi sử dụng Python, PyTorch và Transformers cho xử lý ngôn ngữ tự nhiên, phân loại văn bản. Tôi đã xây dựng tìm kiếm ngữ nghĩa, véc-tơ biểu diễn tài liệu và hệ gợi ý. Tôi triển khai dịch vụ dự đoán của mô hình học máy bằng FastAPI và Docker.


,Rank,Job_ID,Title,Skills,Experience_Years,Domain,Similarity_Score
0,1,J007,Kỹ sư xử lý ngôn ngữ tự nhiên,Python; Xử lý ngôn ngữ tự nhiên (NLP); Transformers; PyTorch; Phân loại văn bản,2,Trí tuệ nhân tạo,0.6378
1,2,J005,Kỹ sư trí tuệ nhân tạo,Python; PyTorch; Transformers; Xử lý ngôn ngữ tự nhiên (NLP); FastAPI,2,Trí tuệ nhân tạo,0.6323
2,3,J006,Kỹ sư học máy,Python; Scikit-learn; PyTorch; Docker; MLflow,3,Trí tuệ nhân tạo,0.2164
3,4,J008,Kỹ sư thị giác máy tính,Python; PyTorch; OpenCV; Học sâu; CNN,2,Trí tuệ nhân tạo,0.2094
4,5,J003,Nhà khoa học dữ liệu,Python; Pandas; SQL; Thống kê; Scikit-learn,2,Trí tuệ nhân tạo,0.1630


### 8. Tự thay CV để thử

Sửa nội dung biến `my_cv` rồi chạy lại ô dưới bằng Shift + Enter.
Với TF-IDF, nên viết CV cùng ngôn ngữ của dữ liệu và ghi rõ kỹ năng thực tế.


In [9]:
my_cv = (
    "Tôi có 2 năm kinh nghiệm lập trình Python, xây dựng REST API bằng FastAPI "
    "và Django, thiết kế cơ sở dữ liệu PostgreSQL, triển khai dịch vụ bằng Docker."
    if LANGUAGE == "vi" else
    "I have 2 years of Python backend experience using FastAPI, Django, PostgreSQL and Docker."
)
display(engine.match_cv_to_jobs(my_cv, top_n=TOP_N)[result_columns].round({"Similarity_Score": 4}))


,Rank,Job_ID,Title,Skills,Experience_Years,Domain,Similarity_Score
0,1,J009,Lập trình viên Backend Python,Python; FastAPI; Django; PostgreSQL; REST API,2,Phát triển phần mềm,0.5193
1,2,J010,Lập trình viên Backend Java,Java; Spring Boot; PostgreSQL; REST API; Docker,2,Phát triển phần mềm,0.2811
2,3,J020,Lập trình viên Backend .NET,C#; .NET; ASP.NET; SQL Server; REST API,2,Phát triển phần mềm,0.2613
3,4,J013,Lập trình viên Full Stack,TypeScript; React; Node.js; PostgreSQL; REST API,3,Phát triển phần mềm,0.2572
4,5,J011,Lập trình viên Backend Node.js,JavaScript; TypeScript; Node.js; Express; PostgreSQL,1,Phát triển phần mềm,0.2502


### 9. Kiểm tra kết quả và lưu bảng

Các kiểm tra dưới đây chỉ xác nhận hành vi phần mềm: loại tin gốc, số dòng và thứ tự
xếp hạng. Chúng không chứng minh độ chính xác gợi ý trên dữ liệu thực và không thay
thế Giai đoạn 3. Kết quả notebook được lưu riêng theo ngôn ngữ và backend.


In [10]:
assert JOB_ID not in similar_jobs["Job_ID"].tolist()
assert len(similar_jobs) == min(TOP_N, len(jobs) - 1)
assert similar_jobs["Similarity_Score"].is_monotonic_decreasing
assert np.isfinite(cv_matches["Similarity_Score"]).all()
for _, group in cv_matches.groupby("Resume_ID"):
    assert group["Similarity_Score"].is_monotonic_decreasing

output_dir = Path("outputs") / "notebook" / LANGUAGE / BACKEND
output_dir.mkdir(parents=True, exist_ok=True)
similar_jobs.to_csv(output_dir / "similar_jobs.csv", index=False, encoding="utf-8-sig")
cv_matches.to_csv(output_dir / "cv_matches.csv", index=False, encoding="utf-8-sig")
print("Các kiểm tra chức năng đều đạt.")
print(f"Đã lưu hai bảng kết quả tại: {output_dir.as_posix()}")


Các kiểm tra chức năng đều đạt.


Đã lưu hai bảng kết quả tại: outputs/notebook/vi/tfidf


### 10. Tùy chọn: Sentence Transformers

Để thử semantic embedding, cài thêm thư viện trong terminal của thư mục dự án:

```powershell
.\.venv\Scripts\python.exe -m pip install -r requirements-semantic.txt
```

Sau đó sửa `BACKEND = "sentence-transformers"` ở ô cấu hình và khởi động lại kernel,
chạy lại tất cả các ô. Lần đầu cần mạng để tải trọng số từ Hugging Face.
Notebook không tự cài thư viện hoặc tự tải model khi chạy mặc định bằng TF-IDF.
Kết quả đã lưu trong bản bàn giao là **TF-IDF**, chưa xác nhận thực thi nhánh semantic.

### Nhận xét để trình bày

- Mock data gồm 20 tin và 3 CV; dữ liệu nhỏ, có chủ đích trùng kỹ năng để minh họa.
- TF-IDF phản ánh độ trùng từ/cụm từ; không tự hiểu mọi từ đồng nghĩa hay dịch giữa các ngôn ngữ.
- Số năm kinh nghiệm chưa là bộ lọc cứng; top-N vẫn có thể chứa việc làm không đủ điều kiện.
- Chưa có ngưỡng chấp nhận; một tin trong top-N có thể có mức phù hợp thấp.
- Bước tiếp theo là xây ground truth và metrics (Giai đoạn 3), sau đó làm Streamlit (Giai đoạn 4).

### Tài liệu tham khảo

- [Jupyter: cài đặt và khởi chạy](https://jupyter.org/install)
- [scikit-learn: TF-IDF](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
- [scikit-learn: cosine similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)
- [Sentence Transformers: mô hình multilingual MiniLM](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)
